# ioNERDSS PDB Validation Suite Tutorial

This tutorial showcases the PDB validation workflow on a real model system, **8ERQ**, and contrasts it with **6BNO**.

The validation suite has three practical steps:

- `setup_simulation(...)`: prepare the validation NERDSS input deck with one copy of each molecule type, titration reactions, and irreversible binding.
- `run_simulation(...)`: run a real NERDSS validation simulation and extract one full assembly if it forms.
- `align_structure(...)`: align the observed coarse-grained structure from NERDSS to the designed target and compute RMSD.


In [ ]:
from pathlib import Path
import json
import tempfile
import numpy as np

from ionerdss.model import pdb
from ionerdss.model.pdb import PDBModelBuilder


## 1. Local NERDSS Executable

This tutorial requires a local NERDSS installation:



In [ ]:
# Replace with your actual NERDSS path!
nerdss_dir = Path('~/Workspace/nerdss_development').expanduser()


## 2. Build the Real 8ERQ System

8ERQ produces three molecule types in the coarse-grained model, so the one-copy-per-type validation target remains meaningful.


In [ ]:
tmp_8erq = tempfile.TemporaryDirectory(prefix='8erq_validation_')
builder_8erq = PDBModelBuilder(source="8erq")
system_8erq = builder_8erq.build_system(
    workspace_path=tmp_8erq.name,
    interface_detect_distance_cutoff=1.0,
    generate_visualizations=False,
    generate_nerdss_files=False,
)


## 3. Prepare the 8ERQ Validation Simulation

We use a moderately small box and a longer run so that the full `A+H+L` assembly has a realistic chance to form during the actual NERDSS simulation.


In [ ]:
artifacts_8erq = pdb.validation.setup_simulation(
    system_8erq,
    workspace_manager=builder_8erq.workspace_manager,
    box_nm=(40.0, 40.0, 40.0),
    initial_molecule_count=1,
    titration_on_rate=2.5e-5,
    parms_overrides={
        'nItr': 200000,
        'timeWrite': 1000,
        'trajWrite': 200000,
        'restartWrite': 200000,
        'checkPoint': 200000,
        'pdbWrite': 200000,
    },
)

with open(artifacts_8erq.target_file, 'r', encoding='utf-8') as handle:
    target_payload_8erq = json.load(handle)

# Validation suite will create a temporary directory
# for the duration of the python session (python using tempfile)
print('Validation counts:', artifacts_8erq.molecule_counts)
print('Target file:', artifacts_8erq.target_file)
print('parms.inp:', artifacts_8erq.nerdss_files['parms'])
print('parms_titrate.inp:', artifacts_8erq.nerdss_files['parms_titrate'])
print('Target payload keys:', target_payload_8erq.keys())


## 4. Show the Generated Titration Encoding

The generated `parms_titrate.inp` titration reactions are encoded with 0th order creation reactions appear before binding reactions using the `0 -> Molecule(site1, site2, ...)` form.


In [ ]:
generated_lines = Path(artifacts_8erq.nerdss_files['parms_titrate']).read_text().splitlines()

generated_titration = [line for line in generated_lines if '0 -> ' in line or 'onRate3Dka' in line][:6]

print('Generated 8ERQ titration snippet:')
print('\n'.join(generated_titration))


## 5. Run the Actual NERDSS Validation Simulation

This is the key validation step: the observed structure is taken from a real NERDSS run, not from a synthetic rigid transform.

If no full assembly matching the designed one-copy target appears in the histogram, the validation suite returns a warning and does not attempt alignment.


In [ ]:
sim_result_8erq = pdb.validation.run_simulation(
    artifacts_8erq,
    nerdss_dir=nerdss_dir,
)

print('Simulation dir:', sim_result_8erq.simulation_dir)
print('Histogram file:', sim_result_8erq.histogram_file)
print('Full assembly found:', sim_result_8erq.full_assembly_found)
print('First full assembly time:', sim_result_8erq.first_full_assembly_time)
if sim_result_8erq.warning_message:
    print(sim_result_8erq.warning_message)
else:
    print('Observed coordinates:', sim_result_8erq.observed_coordinates)


## 6. Align the Observed NERDSS Structure Back to the Design

Only perform alignment when the real NERDSS run actually produced at least one full assembly.


In [ ]:
if sim_result_8erq.full_assembly_found:
    alignment_8erq = pdb.validation.align_structure(
        artifacts_8erq.designed_coordinates,
        sim_result_8erq.observed_coordinates,
        backend='kabsch',
    )
    print('Observed RMSD:', alignment_8erq.rmsd)
else:
    print('Skipping alignment because no full assembly was found in the NERDSS simulation.')
